# Genetic Algorithm - Drone Delivery Route Optimization

This notebook now mirrors the exact code from `genetic.py` and explains each block of the implementation and the result interpretation.

In [ ]:
"""Genetic Algorithm for Drone Delivery Route Optimization."""

from __future__ import annotations

import random
import sys
from pathlib import Path
from time import perf_counter
from typing import Dict, List, Optional, Tuple

if __package__ is None or __package__ == "":
    sys.path.append(str(Path(__file__).resolve().parent))

from solver_base import Problem, greedy_solve

### Imports and module setup

This cell contains the exact import and path-handling logic from `genetic.py`.

- `random` controls selection, crossover, mutation, and seeding.
- `perf_counter` measures runtime.
- `Problem` loads the optimization model.
- `greedy_solve` is used to seed the population with a strong initial permutation.
- The `sys.path` block keeps the file runnable from different execution contexts.

In [ ]:
class GeneticSolver:
    def __init__(
        self,
        problem: Problem,
        pop_size: int = 80,
        generations: int = 300,
        tournament_size: int = 3,
        crossover_rate: float = 0.85,
        mutation_rate: float = 0.15,
        elitism: int = 2,
    ):
        self.problem = problem
        self.pop_size = max(2, pop_size)
        self.generations = max(1, generations)
        self.tournament_size = max(2, min(tournament_size, self.pop_size))
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.elitism = max(0, min(elitism, self.pop_size - 1))

    def fitness(self, perm: List[int]) -> Tuple[float, float, int, List[Tuple[int, List[int]]], set]:
        solution, unserved = self.problem.decode(perm)
        fit, energy, violations, _ = self.problem.evaluate(solution)
        fit += 1000 * len(unserved)
        return fit, energy, violations, solution, unserved

    def create_individual(self) -> List[int]:
        perm = list(range(self.problem.n))
        random.shuffle(perm)
        return perm

    def seed_individual(self) -> List[int]:
        greedy_solution = greedy_solve(self.problem)
        seed_perm: List[int] = []
        for _, route in greedy_solution:
            seed_perm.extend(route)
        served = set(seed_perm)
        for customer in range(self.problem.n):
            if customer not in served:
                seed_perm.append(customer)
        return seed_perm if seed_perm else self.create_individual()

    def tournament_select(
        self,
        population: List[List[int]],
        fits: List[Tuple[float, float, int, List[Tuple[int, List[int]]], set]],
    ) -> List[int]:
        winner: Optional[List[int]] = None
        winner_fit = float("inf")
        for _ in range(self.tournament_size):
            idx = random.randrange(len(population))
            candidate_fit = fits[idx][0]
            if candidate_fit < winner_fit:
                winner_fit = candidate_fit
                winner = population[idx][:]
        return winner if winner is not None else population[0][:]

    def ox_crossover(self, parent1: List[int], parent2: List[int]) -> List[int]:
        n = len(parent1)
        a, b = sorted(random.sample(range(n), 2))
        child: List[Optional[int]] = [None] * n
        child[a:b] = parent1[a:b]
        used = set(parent1[a:b])
        pos = b % n

        for gene in parent2[b:] + parent2[:b]:
            if gene in used:
                continue
            while child[pos] is not None:
                pos = (pos + 1) % n
            child[pos] = gene
            pos = (pos + 1) % n

        return [gene for gene in child if gene is not None]

    def mutate(self, individual: List[int]) -> List[int]:
        n = len(individual)
        if n < 2:
            return individual

        r = random.random()
        if r < 0.33:
            i, j = random.sample(range(n), 2)
            individual[i], individual[j] = individual[j], individual[i]
        elif r < 0.66:
            i, j = random.sample(range(n), 2)
            value = individual.pop(i)
            individual.insert(j, value)
        else:
            a, b = sorted(random.sample(range(n), 2))
            individual[a:b] = list(reversed(individual[a:b]))
        return individual

    def build_initial_population(self) -> List[List[int]]:
        population = [self.create_individual() for _ in range(self.pop_size)]
        population[0] = self.seed_individual()
        if self.pop_size > 1:
            population[1] = list(reversed(population[0]))
        return population

    def solve(self) -> Dict[str, object]:
        population = self.build_initial_population()

        best_perm: Optional[List[int]] = None
        best_solution: Optional[List[Tuple[int, List[int]]]] = None
        best_fitness = float("inf")
        best_energy = float("inf")
        best_violations = 0
        history: List[Dict[str, float]] = []

        for generation in range(self.generations):
            fits = [self.fitness(individual) for individual in population]

            generation_best_idx = min(range(len(population)), key=lambda i: fits[i][0])
            generation_best = fits[generation_best_idx]
            history.append(
                {
                    "generation": float(generation),
                    "fitness": float(generation_best[0]),
                    "energy_kwh": float(generation_best[1]),
                    "violations": float(generation_best[2]),
                    "unserved": float(len(generation_best[4])),
                }
            )

            if generation_best[0] < best_fitness:
                best_fitness = generation_best[0]
                best_energy = generation_best[1]
                best_violations = generation_best[2]
                best_solution = generation_best[3]
                best_perm = population[generation_best_idx][:]

            if generation < self.generations - 1:
                next_population: List[List[int]] = []
                elite_indices = sorted(range(len(population)), key=lambda i: fits[i][0])[: self.elitism]
                for idx in elite_indices:
                    next_population.append(population[idx][:])

                while len(next_population) < self.pop_size:
                    parent1 = self.tournament_select(population, fits)
                    parent2 = self.tournament_select(population, fits)

                    if random.random() < self.crossover_rate:
                        child = self.ox_crossover(parent1, parent2)
                    else:
                        child = parent1[:]

                    if random.random() < self.mutation_rate:
                        child = self.mutate(child)

                    if len(child) != self.problem.n:
                        child = self.create_individual()

                    next_population.append(child)

                population = next_population

            if generation % 50 == 0 or generation == self.generations - 1:
                print(
                    f"Generation {generation:3d}: "
                    f"Best fitness = {best_fitness:10.2f} | "
                    f"Energy = {best_energy:7.3f} kWh | "
                    f"Violations = {best_violations}"
                )

        return {
            "solution": best_solution or [],
            "fitness": best_fitness,
            "energy_kwh": best_energy,
            "violations": best_violations,
            "permutation": best_perm or [],
            "history": history,
            "generations": self.generations,
        }


### `GeneticSolver`

This cell contains the complete genetic algorithm exactly as it appears in `genetic.py`.

Interpretation of the main parts:
- `__init__` validates and stores GA parameters
- `fitness` decodes a permutation and applies penalties for unserved customers
- `create_individual` builds random chromosomes
- `seed_individual` injects the greedy route as a strong starting point
- `tournament_select` chooses parents
- `ox_crossover` preserves ordering for permutation problems
- `mutate` applies one of three local changes
- `build_initial_population` creates diversity with a greedy seed and its reverse
- `solve` runs the evolutionary loop and records history

In [ ]:
def summarize_solution(problem: Problem, solution: List[Tuple[int, List[int]]]) -> Dict[str, float]:
    fitness, energy, violations, unserved = problem.evaluate(solution)
    return {
        "fitness": float(fitness),
        "energy_kwh": float(energy),
        "violations": int(violations),
        "unserved": int(len(unserved)),
        "served": int(problem.n - len(unserved)),
        "routes": int(sum(1 for _, route in solution if route)),
    }


### `summarize_solution`

This helper makes the final genetic solution comparable to the greedy solution.

It reports the same metrics as the greedy notebook:
- fitness
- energy
- violations
- unserved
- served
- routes

That shared format is what makes the comparative analysis possible.

In [ ]:
def run_genetic(
    seed: int = 42,
    pop_size: int = 80,
    generations: int = 300,
    tournament_size: int = 3,
    crossover_rate: float = 0.85,
    mutation_rate: float = 0.15,
    elitism: int = 2,
) -> Dict[str, object]:
    random.seed(seed)
    problem = Problem()

    solver = GeneticSolver(
        problem,
        pop_size=pop_size,
        generations=generations,
        tournament_size=tournament_size,
        crossover_rate=crossover_rate,
        mutation_rate=mutation_rate,
        elitism=elitism,
    )

    start = perf_counter()
    result = solver.solve()
    elapsed = perf_counter() - start

    metrics = summarize_solution(problem, result["solution"])
    metrics["runtime_sec"] = elapsed

    return {
        "problem": problem,
        "solution": result["solution"],
        "metrics": metrics,
        "history": result["history"],
        "fitness": result["fitness"],
        "permutation": result["permutation"],
    }


### `run_genetic`

This cell reproduces the full execution wrapper from `genetic.py`.

Interpretation:
- it seeds randomness for reproducibility
- it creates the `Problem`
- it builds the genetic solver with the chosen parameters
- it runs the optimization and measures runtime
- it returns the problem, solution, metrics, history, fitness, and permutation

In [ ]:
def print_report(problem: Problem, solution: List[Tuple[int, List[int]]], metrics: Dict[str, float]) -> None:
    print("=" * 70)
    print("GENETIC ALGORITHM - Drone Delivery Route Optimization")
    print("=" * 70)
    print(problem.format(solution))
    print("-" * 70)
    print(f"Total Energy:          {metrics['energy_kwh']:.3f} kWh")
    print(f"Constraint Violations: {metrics['violations']}")
    print(f"Unserved Customers:    {metrics['unserved']}")
    print(f"Routes Used:           {metrics['routes']}")
    print(f"Runtime:               {metrics['runtime_sec']:.4f} s")
    if metrics["unserved"]:
        _, _, _, unserved = problem.evaluate(solution)
        print(f"  -> {[problem.customers[i]['name'] for i in sorted(unserved)]}")
    print("=" * 70)


def main() -> Dict[str, object]:
    result = run_genetic()
    problem = result["problem"]
    solution = result["solution"]
    metrics = result["metrics"]
    print_report(problem, solution, metrics)
    return result


if __name__ == "__main__":
    main()


### `print_report`, `main`, and script guard

This final code cell keeps the exact reporting and entry-point behavior from `genetic.py`.

Interpretation:
- `print_report` displays the final best route and summary metrics
- `main()` runs the complete genetic pipeline
- the script guard allows the file to run as a standalone module

The result should be interpreted as a quality-vs-runtime tradeoff: the genetic method is slower, but it can reduce energy consumption compared with the greedy baseline.